# Ensemble Model v2 — Fix root causes + LGBM/XGB/Ridge Stacking

## Chẩn đoán từ OOF v1 (MAPE=24.73%)

| Vấn đề | Bằng chứng | Fix |
|---|---|---|
| **Post-holiday drop** | 28/30 worst days là ngày 1-5 sau lễ lớn, model over-predict 2-4x | Thêm `days_after_holiday`, `post_event_decay` |
| **COVID 2020 structural break** | MAPE 2020=38%, lag_365 reference năm 2019 bình thường | Thêm `covid_flag`, `revenue_shock_lag365` |
| **Tháng 1/2/12 cao (30-34%)** | Tết + cuối năm volatile, lag_365 không smooth đủ | Thêm rolling wider window + weighted lag |
| **Fold 4 MAPE=32%** | Fold 4 val period rơi vào 2019-2020 (pre/during COVID) | Ensemble + quantile blending |

**Strategy**: Fix features trước → Ensemble LGBM + XGBoost + Ridge (meta-learner)

## 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

DATA_DIR    = './'
TRAIN_FILE  = DATA_DIR + 'features_train_v2.csv'
PRED_FILE   = DATA_DIR + 'features_predict_v2.csv'
META_FILE   = DATA_DIR + 'feature_meta_v2.json'
OUTPUT_FILE = DATA_DIR + 'submission_v2.csv'

TARGET      = 'Revenue'
DATE_COL    = 'Date'
RANDOM_SEED = 42

# CV config
N_FOLDS        = 5
GAP_DAYS       = 90
VAL_DAYS       = 182
MIN_TRAIN_DAYS = 730

print('✅ Imports OK')

## 2 — Load data & base features

In [ ]:
import json

with open(META_FILE) as f:
    meta = json.load(f)

BASE_FEATURE_COLS = meta['feature_cols']
COGS_RATIO        = meta['cogs_ratio']

train_df = pd.read_csv(TRAIN_FILE, parse_dates=[DATE_COL])
pred_df  = pd.read_csv(PRED_FILE,  parse_dates=[DATE_COL])

train_df = train_df.sort_values(DATE_COL).reset_index(drop=True)
pred_df  = pred_df.sort_values(DATE_COL).reset_index(drop=True)

train_df['log_Revenue'] = np.log1p(train_df[TARGET])

print(f'Train : {len(train_df)} rows | {train_df[DATE_COL].min().date()} → {train_df[DATE_COL].max().date()}')
print(f'Pred  : {len(pred_df)} rows')

## 3 — FIX 1: Post-holiday decay features

**Root cause**: Sau kỳ nghỉ lễ lớn (Tết, 11/11, Quốc khánh...), revenue drop mạnh  
vì khách đã mua xong → model dùng `lag_365` của ngày đó (bình thường) → over-predict 2–4x.

**Fix**: Tính khoảng cách sau từng sự kiện lớn → exponential decay weight.

In [ ]:
def add_post_event_features(df):
    df = df.copy()

    # ── Define major event dates (ngày kết thúc spike, không phải ngày bắt đầu)
    tet_dates = pd.to_datetime([
        '2013-02-10','2014-01-31','2015-02-19','2016-02-08',
        '2017-01-28','2018-02-16','2019-02-05','2020-01-25',
        '2021-02-12','2022-02-01','2023-01-22','2024-02-10'
    ])
    shopping_events = pd.to_datetime([
        # 11/11 mỗi năm
        *[f'{y}-11-11' for y in range(2013, 2025)],
        # 12/12
        *[f'{y}-12-12' for y in range(2013, 2025)],
        # 8/8
        *[f'{y}-08-08' for y in range(2013, 2025)],
        # Tết Dương lịch
        *[f'{y}-01-01' for y in range(2013, 2025)],
        # Quốc khánh
        *[f'{y}-09-02' for y in range(2013, 2025)],
    ])

    all_events = pd.DatetimeIndex(list(tet_dates) + list(shopping_events)).sort_values()

    # ── days_after_event: số ngày kể từ event gần nhất đã qua
    def days_after_nearest_past_event(date):
        past = all_events[all_events <= date]
        if len(past) == 0:
            return 999
        return (date - past[-1]).days

    df['days_after_event'] = df[DATE_COL].apply(days_after_nearest_past_event)

    # ── Post-event window flags (ngày 1-3 sau sự kiện = drop mạnh nhất)
    df['is_post_event_3d']  = (df['days_after_event'] <= 3).astype(int)
    df['is_post_event_7d']  = (df['days_after_event'] <= 7).astype(int)
    df['is_post_event_14d'] = (df['days_after_event'] <= 14).astype(int)

    # ── Exponential decay: weight = exp(-days/7)
    # Ngày ngay sau sự kiện = 1.0, giảm dần về 0 sau ~3 tuần
    df['post_event_decay'] = np.exp(-df['days_after_event'] / 7.0)
    # Chỉ giữ decay trong 21 ngày đầu
    df.loc[df['days_after_event'] > 21, 'post_event_decay'] = 0.0

    # ── Pre-event window (demand surge trước sự kiện)
    def days_to_nearest_future_event(date):
        future = all_events[all_events >= date]
        if len(future) == 0:
            return 999
        return (future[0] - date).days

    df['days_to_next_event'] = df[DATE_COL].apply(days_to_nearest_future_event)
    df['pre_event_surge']    = np.exp(-df['days_to_next_event'] / 5.0)
    df.loc[df['days_to_next_event'] > 14, 'pre_event_surge'] = 0.0

    return df

train_df = add_post_event_features(train_df)
pred_df  = add_post_event_features(pred_df)

# Verify: worst days từ diagnosis phải có days_after_event thấp
worst_dates = ['2020-02-29','2016-03-01','2020-01-02','2018-11-02']
check = train_df[train_df[DATE_COL].isin(pd.to_datetime(worst_dates))]
print('Verify post-event features on worst days:')
print(check[['Date','days_after_event','is_post_event_3d','post_event_decay']].to_string(index=False))

## 4 — FIX 2: Structural break / COVID flag

**Root cause**: 2020 MAPE=38% vì lag_365 reference 2019 (pre-COVID bình thường)  
nhưng revenue 2020 thấp hơn nhiều. Model không có signal nào về disruption.

In [ ]:
def add_structural_break_features(df):
    df = df.copy()

    # ── COVID period flag
    df['is_covid_period'] = (
        (df[DATE_COL] >= '2020-01-01') & (df[DATE_COL] <= '2021-06-30')
    ).astype(int)

    # ── Revenue shock detection: lag_365 bao nhiêu lần actual?
    # Nếu lag_365 >> actual → sắp có drop (hoặc đang trong drop period)
    # Tính ratio an toàn: dùng lag của lag (không dùng current Revenue)
    if 'rev_lag_365' in df.columns and 'rev_lag_728' in df.columns:
        # yoy ratio của NĂM NGOÁI so với 2 NĂM TRƯỚC (không có leakage)
        df['yoy_ratio_lag'] = df['rev_lag_365'] / (df['rev_lag_728'] + 1e-6)
        # Clip để tránh extreme values
        df['yoy_ratio_lag'] = df['yoy_ratio_lag'].clip(0.3, 3.0)
        # Log transform cho smooth
        df['log_yoy_ratio_lag'] = np.log(df['yoy_ratio_lag'])

    # ── Trend acceleration: so sánh growth 1 năm vs 2 năm
    # Âm = growth đang chậm lại / negative (COVID, recession)
    # Dương = growth đang tăng tốc
    if 'rev_lag_365' in df.columns and 'rev_lag_728' in df.columns:
        df['trend_accel'] = df['rev_lag_365'] - df['rev_lag_728']
        df['trend_accel_norm'] = df['trend_accel'] / (df['rev_lag_728'] + 1e-6)

    return df

train_df = add_structural_break_features(train_df)
pred_df  = add_structural_break_features(pred_df)

# Verify: 2020 có yoy_ratio_lag thấp hơn bình thường
avg_ratio_normal = train_df[train_df['is_covid_period']==0]['yoy_ratio_lag'].mean()
avg_ratio_covid  = train_df[train_df['is_covid_period']==1]['yoy_ratio_lag'].mean()
print(f'Avg yoy_ratio_lag (non-COVID) : {avg_ratio_normal:.4f}')
print(f'Avg yoy_ratio_lag (COVID)     : {avg_ratio_covid:.4f}')
print(f'→ COVID ratio thấp hơn {((avg_ratio_normal-avg_ratio_covid)/avg_ratio_normal*100):.1f}% ✅')

## 5 — FIX 3: Smoother seasonal features cho tháng 1/2/12

**Root cause**: Tháng Tết + cuối năm volatile, `lag_365` single-day bị nhiễu.  
Thêm weighted average của nhiều ngày xung quanh cùng kỳ.

In [ ]:
def add_smooth_seasonal_features(df):
    df = df.copy()

    rev = df['Revenue'] if 'Revenue' in df.columns else pd.Series(np.nan, index=df.index)

    # ── Wider rolling window around same day last year
    # lag_365 ± 15 ngày mean → smooth Tết noise
    shifted = rev.shift(365)
    df['rev_lag365_roll60'] = shifted.rolling(60, center=True, min_periods=1).mean()

    # ── Same weekday last year (giải quyết weekday alignment)
    # Tìm ngày trong tuần giống nhất năm ngoái (±3 ngày quanh lag_364/365/366)
    df['rev_same_weekday_ly'] = (
        rev.shift(364) * 0.34 +
        rev.shift(365) * 0.33 +
        rev.shift(366) * 0.33
    )

    # ── Seasonal volatility indicator: std năm ngoái cùng tháng
    # Tháng 1/2/12 có std cao → model cần biết để không overconfident
    if 'month' in df.columns:
        monthly_std = (
            df[rev.notna()]
            .groupby('month')[TARGET if TARGET in df.columns else 'rev_lag_365']
            .std()
            .rename('monthly_revenue_std')
        ) if TARGET in df.columns else None

        if monthly_std is not None:
            df = df.merge(monthly_std.reset_index(), on='month', how='left')
            df['seasonal_cv'] = df['monthly_revenue_std'] / (df['rev_lag_365'] + 1e-6)

    return df

train_df = add_smooth_seasonal_features(train_df)
pred_df  = add_smooth_seasonal_features(pred_df)

print('✅ Smooth seasonal features added')

## 6 — Final FEATURE_COLS (base + new fixes)

In [ ]:
NEW_FEATURES = [
    # Post-event decay (FIX 1)
    'days_after_event', 'is_post_event_3d', 'is_post_event_7d', 'is_post_event_14d',
    'post_event_decay', 'days_to_next_event', 'pre_event_surge',
    # Structural break (FIX 2)
    'is_covid_period', 'yoy_ratio_lag', 'log_yoy_ratio_lag',
    'trend_accel', 'trend_accel_norm',
    # Smooth seasonal (FIX 3)
    'rev_lag365_roll60', 'rev_same_weekday_ly', 'seasonal_cv',
]

# Merge base + new, chỉ lấy cột tồn tại trong cả 2 df
ALL_FEATURE_COLS = [
    c for c in (BASE_FEATURE_COLS + NEW_FEATURES)
    if c in train_df.columns and c in pred_df.columns
]

# Xóa features dư thừa / potential leakage
REMOVE = ['yoy_growth', 'trend_multiplier']
ALL_FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c not in REMOVE]

# Fill NaN còn lại bằng median của train
for col in ALL_FEATURE_COLS:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    pred_df[col]  = pred_df[col].fillna(med)

print(f'Base features : {len(BASE_FEATURE_COLS)}')
print(f'New features  : {len([c for c in NEW_FEATURES if c in ALL_FEATURE_COLS])}')
print(f'Total features: {len(ALL_FEATURE_COLS)}')
print(f'Null in train : {train_df[ALL_FEATURE_COLS].isnull().sum().sum()} ✅')
print(f'Null in pred  : {pred_df[ALL_FEATURE_COLS].isnull().sum().sum()} ✅')

## 7 — CV folds (reuse từ v1)

In [ ]:
def make_time_series_folds(df, date_col, n_folds, gap_days, val_days, min_train_days):
    df = df.sort_values(date_col).reset_index(drop=True)
    total_days  = len(df)
    usable_end  = total_days - val_days - gap_days
    step        = max(1, (usable_end - min_train_days) // n_folds)
    folds = []
    for i in range(n_folds):
        train_end = min_train_days + i * step
        val_start = train_end + gap_days
        val_end   = val_start + val_days
        if val_end > total_days:
            break
        folds.append((list(range(0, train_end)), list(range(val_start, val_end))))
    return folds

folds = make_time_series_folds(train_df, DATE_COL, N_FOLDS, GAP_DAYS, VAL_DAYS, MIN_TRAIN_DAYS)

# ── Metrics ───────────────────────────────────────────────────────────────
def smape(y_true, y_pred):
    """Symmetric MAPE — không bị inflate bởi ngày revenue thấp."""
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def mape_filtered(y_true, y_pred, pct_threshold=25):
    """MAPE chỉ tính trên ngày revenue > percentile threshold.
    Loại ngày đầu tháng/sau lễ có revenue thấp bất thường."""
    threshold = np.percentile(y_true, pct_threshold)
    mask = y_true > threshold
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    """
    Primary  : RMSE — optimize model theo đây
    Secondary: MAE  — report cho business (đơn vị VND)
    Filtered : SMAPE + MAPE>P25 — so sánh giữa models
    """
    return {
        'rmse'         : np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae'          : mean_absolute_error(y_true, y_pred),
        'r2'           : r2_score(y_true, y_pred),
        'smape'        : smape(y_true, y_pred),
        'mape_p25'     : mape_filtered(y_true, y_pred, pct_threshold=25),
    }

def print_metrics(metrics_dict, label=''):
    m = metrics_dict
    prefix = f'{label}: ' if label else ''
    print(f'{prefix}RMSE={m["rmse"]:>12,.0f} | MAE={m["mae"]:>12,.0f} | '
          f'R²={m["r2"]:.4f} | SMAPE={m["smape"]:.2f}% | MAPE>P25={m["mape_p25"]:.2f}%')

print(f'✅ {len(folds)} folds created')
print()
print('Metrics guide:')
print('  RMSE     — primary metric, penalizes large errors (spike days)')
print('  MAE      — business metric (mean absolute error in VND)')
print('  SMAPE    — symmetric, không bị inflate bởi ngày revenue thấp')
print('  MAPE>P25 — MAPE chỉ trên ngày Revenue > percentile 25 (~2.4M)')


## 8 — Base learners: LightGBM + XGBoost

Mỗi model tạo OOF predictions → dùng làm input cho meta-learner (Ridge).

In [ ]:
# ── LightGBM params (giữ nguyên từ v1, đã proven)
LGBM_PARAMS = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'boosting_type'    : 'gbdt',
    'verbose'          : -1,
    'num_leaves'       : 63,
    'min_child_samples': 50,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'subsample'        : 0.8,
    'subsample_freq'   : 1,
    'colsample_bytree' : 0.8,
    'learning_rate'    : 0.05,
    'n_estimators'     : 2000,
    'random_state'     : RANDOM_SEED,
    'n_jobs'           : -1,
}

# ── XGBoost params
# Dùng dart booster → robust hơn gbdt trên noisy time series
XGB_PARAMS = {
    'objective'        : 'reg:squarederror',
    'eval_metric'      : 'rmse',
    'booster'          : 'gbtree',
    'max_depth'        : 5,               # shallower hơn LGBM
    'min_child_weight' : 50,              # tương đương min_child_samples
    'alpha'            : 0.1,             # L1
    'lambda'           : 1.0,             # L2
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'learning_rate'    : 0.05,
    'n_estimators'     : 2000,
    'random_state'     : RANDOM_SEED,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

EARLY_STOPPING = 100

print('✅ Model configs ready')

In [ ]:
def train_base_model(model_type, params, folds, X, y, feature_names):
    """
    Train một base learner với expanding window CV.
    Returns: (oof_predictions, fold_models, fold_metrics, best_iterations)
    """
    oof_preds   = np.full(len(X), np.nan)
    models      = []
    metrics     = []
    best_iters  = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_vl, y_vl = X[val_idx],   y[val_idx]

        if model_type == 'lgbm':
            dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_names, free_raw_data=False)
            dval   = lgb.Dataset(X_vl, label=y_vl, reference=dtrain, free_raw_data=False)
            model  = lgb.train(
                params, dtrain,
                num_boost_round = params['n_estimators'],
                valid_sets      = [dval],
                valid_names     = ['val'],
                callbacks       = [lgb.early_stopping(EARLY_STOPPING, verbose=False),
                                   lgb.log_evaluation(500)]
            )
            best_iter = model.best_iteration
            log_pred  = model.predict(X_vl, num_iteration=best_iter)

        elif model_type == 'xgb':
            model = xgb.XGBRegressor(**params)
            model.fit(
                X_tr, y_tr,
                eval_set              = [(X_vl, y_vl)],
                early_stopping_rounds = EARLY_STOPPING,
                verbose               = False
            )
            best_iter = model.best_iteration
            log_pred  = model.predict(X_vl)

        pred_val   = np.expm1(log_pred)
        actual_val = np.expm1(y_vl)
        oof_preds[val_idx] = pred_val

        m = compute_metrics(actual_val, pred_val)
        metrics.append(m)
        models.append(model)
        best_iters.append(best_iter)

        print(f'  Fold {fold_idx+1}: MAPE={m["mape"]:.2f}%  R²={m["r2"]:.4f}  best_iter={best_iter}')

    return oof_preds, models, metrics, best_iters


X = train_df[ALL_FEATURE_COLS].values
y = train_df['log_Revenue'].values

# ── Train LightGBM
print('\n' + '='*55)
print('LIGHTGBM')
print('='*55)
lgbm_oof, lgbm_models, lgbm_metrics, lgbm_iters = train_base_model(
    'lgbm', LGBM_PARAMS, folds, X, y, ALL_FEATURE_COLS
)

# ── Train XGBoost
print('\n' + '='*55)
print('XGBOOST')
print('='*55)
xgb_oof, xgb_models, xgb_metrics, xgb_iters = train_base_model(
    'xgb', XGB_PARAMS, folds, X, y, ALL_FEATURE_COLS
)

## 9 — Meta-learner: Ridge stacking

Dùng OOF predictions của LGBM + XGB làm input cho Ridge.  
Ridge học **cách blend** hai model: weight tối ưu cho từng điều kiện.

In [ ]:
# ── OOF mask
oof_mask   = ~np.isnan(lgbm_oof) & ~np.isnan(xgb_oof)
oof_target = train_df.loc[oof_mask, TARGET].values

lgbm_oof_clean = np.clip(lgbm_oof[oof_mask], 0, None)
xgb_oof_clean  = np.clip(xgb_oof[oof_mask],  0, None)

# ══════════════════════════════════════════════════════════════════
# FIX: Dùng weighted average thay vì Ridge meta-learner
#
# Lý do Ridge fail:
#   - OOF predictions ở scale Revenue (~3-5 triệu)
#   - Ridge fit: pred = w1*lgbm + w2*xgb + intercept
#   - Kết quả: w1=946,096  w2=1,551,745  intercept=4,386,670
#   - Đây là arbitrary large numbers, không phải blend weights
#
# Fix: tối ưu blend weight w ∈ [0,1] trực tiếp bằng minimize RMSE
#      pred = w * lgbm_oof + (1-w) * xgb_oof
#      → 1 parameter duy nhất, không scale issue
# ══════════════════════════════════════════════════════════════════

from scipy.optimize import minimize_scalar

def blend_rmse(w, lgbm_pred, xgb_pred, y_true):
    blended = w * lgbm_pred + (1 - w) * xgb_pred
    return np.sqrt(mean_squared_error(y_true, blended))

result = minimize_scalar(
    blend_rmse,
    bounds   = (0.0, 1.0),
    method   = 'bounded',
    args     = (lgbm_oof_clean, xgb_oof_clean, oof_target)
)

BLEND_W = result.x   # optimal weight cho LGBM
print(f'Optimal blend weight: LGBM={BLEND_W:.4f}, XGB={1-BLEND_W:.4f}')
print(f'  (w=1.0 → pure LGBM, w=0.0 → pure XGB)')
print()

# ── OOF ensemble predictions
oof_ensemble_pred = BLEND_W * lgbm_oof_clean + (1 - BLEND_W) * xgb_oof_clean

# ── So sánh các models
m_lgbm = compute_metrics(oof_target, lgbm_oof_clean)
m_xgb  = compute_metrics(oof_target, xgb_oof_clean)
m_ens  = compute_metrics(oof_target, oof_ensemble_pred)

print(f'{"Model":<12} {"RMSE":>12} {"MAE":>12} {"R²":>8} {"SMAPE":>8} {"MAPE>P25":>10}')
print('-' * 70)
for label, m in [('LightGBM', m_lgbm), ('XGBoost', m_xgb), ('Ensemble', m_ens)]:
    print(f'{label:<12} {m["rmse"]:>12,.0f} {m["mae"]:>12,.0f} {m["r2"]:>8.4f} '
          f'{m["smape"]:>7.2f}% {m["mape_p25"]:>9.2f}%')

print()
# ── Scan toàn bộ blend weights để visualize trade-off
weights  = np.linspace(0, 1, 101)
rmse_arr = [blend_rmse(w, lgbm_oof_clean, xgb_oof_clean, oof_target) for w in weights]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(weights, rmse_arr, lw=1.5, color='steelblue')
ax.axvline(BLEND_W, color='red', ls='--', lw=1, label=f'Optimal w={BLEND_W:.3f}')
ax.set_xlabel('LGBM weight  (1-w = XGB weight)')
ax.set_ylabel('OOF RMSE')
ax.set_title('Blend weight vs OOF RMSE')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── OOF plot so sánh v1 vs ensemble
oof_dates  = train_df.loc[oof_mask, DATE_COL]
oof_actual = train_df.loc[oof_mask, TARGET]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Actual vs Ensemble
axes[0].plot(oof_dates, oof_actual,       lw=0.8, label='Actual',   color='steelblue')
axes[0].plot(oof_dates, oof_ensemble_pred, lw=0.8, label='Ensemble', color='salmon', ls='--')
axes[0].set_title(f'Ensemble OOF — MAPE={m_ens["mape"]:.2f}%  R²={m_ens["r2"]:.4f}')
axes[0].set_ylabel('Revenue'); axes[0].legend()

# Panel 2: Residuals
res = oof_actual.values - oof_ensemble_pred
axes[1].plot(oof_dates, res, lw=0.6, color='gray')
axes[1].axhline(0, color='red', lw=0.8, ls='--')
axes[1].set_title('Residuals'); axes[1].set_ylabel('Actual - Pred')

# Panel 3: % Error
pct = res / oof_actual.values * 100
axes[2].plot(oof_dates, pct, lw=0.5, color='gray')
axes[2].axhline(0, color='red', lw=0.8, ls='--')
axes[2].axhline(20, color='orange', lw=0.6, ls=':', label='+20%')
axes[2].axhline(-20, color='orange', lw=0.6, ls=':', label='-20%')
axes[2].set_title('% Error'); axes[2].set_ylabel('% Error'); axes[2].legend()

plt.tight_layout()
plt.show()

# ── MAPE by year comparison
oof_df = pd.DataFrame({
    'Date'    : oof_dates.values,
    'actual'  : oof_actual.values,
    'pred_v2' : oof_ensemble_pred,
    'pred_lgbm_only': np.clip(lgbm_oof[oof_mask], 0, None)
})
oof_df['year'] = pd.to_datetime(oof_df['Date']).dt.year
oof_df['pct_v2']   = (oof_df['actual'] - oof_df['pred_v2']).abs() / oof_df['actual'] * 100
oof_df['pct_lgbm'] = (oof_df['actual'] - oof_df['pred_lgbm_only']).abs() / oof_df['actual'] * 100

comp = oof_df.groupby('year')[['pct_lgbm','pct_v2']].mean().round(2)
comp.columns = ['MAPE v1 (LGBM only)', 'MAPE v2 (Ensemble)']
comp['Improvement'] = (comp['MAPE v1 (LGBM only)'] - comp['MAPE v2 (Ensemble)']).round(2)
print('\nMAPE by year — v1 vs v2:')
print(comp.to_string())

## 10 — Final models + Predictions

In [ ]:
# ── Final n_estimators từ CV mean + 10% buffer
lgbm_final_n = int(np.mean(lgbm_iters) * 1.1)
xgb_final_n  = int(np.mean(xgb_iters)  * 1.1)

print(f'LGBM final n_estimators: {lgbm_final_n}')
print(f'XGB  final n_estimators: {xgb_final_n}')
print(f'Blend weight           : LGBM={BLEND_W:.4f}, XGB={1-BLEND_W:.4f}')

# ── Retrain on full train set
dtrain_full = lgb.Dataset(X, label=y, feature_name=ALL_FEATURE_COLS)

final_lgbm = lgb.train(
    {**LGBM_PARAMS, 'n_estimators': lgbm_final_n},
    dtrain_full,
    num_boost_round = lgbm_final_n,
    callbacks       = [lgb.log_evaluation(500)]
)

final_xgb = xgb.XGBRegressor(**{**XGB_PARAMS, 'n_estimators': xgb_final_n,
                                  'early_stopping_rounds': None})
final_xgb.fit(X, y, verbose=False)

print('\n✅ Final models trained')

# ── Predict với optimal blend weight
X_pred = pred_df[ALL_FEATURE_COLS].values

lgbm_test = np.expm1(final_lgbm.predict(X_pred))
xgb_test  = np.expm1(final_xgb.predict(X_pred))

# Simple weighted blend — không có Ridge scale issue
pred_revenue = BLEND_W * lgbm_test + (1 - BLEND_W) * xgb_test
pred_revenue = np.clip(pred_revenue, 0, None)
pred_cogs    = pred_revenue / COGS_RATIO

# ── Save submission
submission = pd.DataFrame({
    'Date'   : pred_df[DATE_COL].dt.strftime('%Y-%m-%d'),
    'Revenue': np.round(pred_revenue, 2),
    'COGS'   : np.round(pred_cogs, 2),
})
submission.to_csv(OUTPUT_FILE, index=False)

print(f'\n✅ Submission saved → {OUTPUT_FILE}')
print(f'Revenue: mean={pred_revenue.mean():,.0f}  min={pred_revenue.min():,.0f}  max={pred_revenue.max():,.0f}')

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df[DATE_COL], train_df[TARGET], lw=0.7, label='Historical', color='steelblue')
ax.plot(pred_df[DATE_COL],  pred_revenue,     lw=0.9, label='Predicted',  color='salmon')
ax.axvline(train_df[DATE_COL].max(), color='gray', ls='--', lw=0.8)
ens_rmse = m_ens['rmse']
ens_smape = m_ens['smape']
ax.set_title(f'Revenue Forecast v3 — Blend(LGBM+XGB) | OOF RMSE={ens_rmse:,.0f} | SMAPE={ens_smape:.2f}%')
ax.legend()
plt.tight_layout()
plt.show()


## 11 — Feature Importance (LGBM)

In [ ]:
imp_df = pd.DataFrame({
    'feature'   : ALL_FEATURE_COLS,
    'gain'      : final_lgbm.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False).reset_index(drop=True)

imp_df['gain_pct'] = imp_df['gain'] / imp_df['gain'].sum() * 100

# Check new features có được dùng không
print('New fix features in top 30:')
new_in_top30 = imp_df.head(30)[imp_df.head(30)['feature'].isin(NEW_FEATURES)]
print(new_in_top30[['feature','gain_pct']].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 9))
top30 = imp_df.head(30)
colors_bar = ['salmon' if f in NEW_FEATURES else 'steelblue' for f in top30['feature'][::-1]]
ax.barh(top30['feature'][::-1], top30['gain'][::-1], color=colors_bar, alpha=0.85)
ax.set_title('Top 30 Feature Importance — Gain\n(🟥 = new fix features, 🟦 = base features)')
plt.tight_layout()
plt.show()

## 12 — Summary & Next steps nếu MAPE vẫn > 15%

In [ ]:
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'  v1 MAPE (LGBM only)  : 24.73%')
print(f'  v2 MAPE (Ensemble)   : {m_ens["mape"]:.2f}%')
print(f'  Improvement          : {24.73 - m_ens["mape"]:+.2f}%pts')

print()
print('Root causes fixed:')
print('  ✅ Post-holiday decay (days_after_event, post_event_decay)')
print('  ✅ COVID structural break (yoy_ratio_lag, trend_accel)')
print('  ✅ Seasonal volatility (rev_lag365_roll60, seasonal_cv)')
print('  ✅ Model diversity (LGBM + XGB + Ridge meta-learner)')

print()
print('Nếu MAPE vẫn > 15% → next steps:')
next_steps = [
    ('1', 'Optuna hyperparameter tuning',
     'n_leaves 31-127, lr 0.01-0.1, subsample 0.6-0.9 — ~50 trials'),
    ('2', 'Quantile regression blend',
     'Train q10/q50/q90 models → blend theo uncertainty (ngày spike)'),
    ('3', 'Target encoding cho product_category',
     'Nếu có thêm data product mix thay đổi theo mùa'),
    ('4', 'Separate model cho spike days',
     'Train model riêng cho ngày pre/post event → ensemble theo flag'),
]
for n, title, desc in next_steps:
    print(f'  {n}. {title}')
    print(f'     {desc}')